
# S2-standalone — trajectory-only null gate using Q outputs

Purpose: test whether the target-material single-isotherm climate ranking retains information beyond:

1. supported-transition count \(N\), and
2. the shared RH trajectory alone.

This notebook **does not rerun Q Monte Carlo or R stress tests**.

It reuses:
- `nested_climate_scores.csv` from Q for the already-computed full/single climate scores;
- the original climate file;
- the equilibrium master only to reconstruct the **exact Q common-support transition mask** needed for the RH-only null.

Why the master is still needed:
`nested_climate_scores.csv` stores climate-level scores but not transition IDs. To construct
\[
H_{\rm RH}=\frac12\sum|\Delta RH|
\]
on exactly the same transitions used by Q, the branchwise finite-support intersection of full, single, nested-two-slice, and three-slice representations has to be reconstructed once. This is cheap and avoids rerunning the expensive Q/R analyses.

Primary statistic:
\[
\Delta\rho
=
\rho(H_{\rm single},H_{\rm full}\mid N)
-
\rho(H_{\rm RH-null},H_{\rm full}\mid N).
\]

A material × branch × temporal-mode gate passes when the paired climate-bootstrap 95% CI of \(\Delta\rho\) is above zero.


In [ ]:

# 0. Imports, Drive paths, and required files

from google.colab import drive
from google.colab import files as colab_files
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.interpolate import LinearNDInterpolator, interp1d
from scipy.spatial import Delaunay
from scipy.stats import spearmanr, kendalltau, rankdata
from tqdm.auto import tqdm

drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive')
preferred = DRIVE/'climate_screening_v1_3_1_uncertainty_audit'/'climate_screening_backbone'

if preferred.exists():
    PROJECT = preferred
else:
    hits = list(DRIVE.rglob('climate_screening_backbone'))
    if not hits:
        raise FileNotFoundError('climate_screening_backbone not found on Drive.')
    PROJECT = hits[0]

QDIR = PROJECT/'results/Q_confirmatory_nested_uncertainty_timescale'
OUT = PROJECT/'results/S2_trajectory_null_gate_standalone'
OUT.mkdir(parents=True, exist_ok=True)

CLIMATE = PROJECT/'data/processed/climate_hourly_2015_2024.csv'
PEI = PROJECT/'data/raw/PEI_climate_measured_capacity.csv'

# Q output used as the authoritative full/single climate-score source.
SCORES = QDIR/'nested_climate_scores.csv'
if not SCORES.exists():
    print('Upload nested_climate_scores.csv from Q')
    up = colab_files.upload()
    if 'nested_climate_scores.csv' in up:
        SCORES = Path('/content/nested_climate_scores.csv')
    else:
        cand = [n for n in up if 'nested_climate_scores' in n and n.lower().endswith('.csv')]
        if not cand:
            raise FileNotFoundError('nested_climate_scores.csv is required.')
        SCORES = Path('/content')/cand[0]

# Master is used only to reproduce the exact finite-support mask.
MASTER = None
for pat in [
    'q_RH_T_master_cycle08_audited_20261010(1).xlsx',
    'q_RH_T_master_cycle08_audited_20261010.xlsx',
    'q_RH_T_master_combined_v2(1).xlsx',
    'q_RH_T_master_combined_v2.xlsx',
]:
    hits = list(DRIVE.rglob(pat))
    if hits:
        MASTER = hits[0]
        break

if MASTER is None:
    print('Upload the equilibrium master workbook used for Q.')
    up = colab_files.upload()
    xls = [n for n in up if n.lower().endswith(('.xlsx','.xls'))]
    if not xls:
        raise FileNotFoundError('Equilibrium master workbook is required.')
    MASTER = Path('/content')/xls[0]

if not CLIMATE.exists():
    raise FileNotFoundError(f'Missing climate file: {CLIMATE}')

print('PROJECT :', PROJECT)
print('SCORES  :', SCORES)
print('MASTER  :', MASTER)
print('CLIMATE :', CLIMATE)
print('OUT     :', OUT)


## 1. Load only the data needed to reconstruct Q support

In [ ]:

def normalize_columns(df):
    d = df.copy()
    ren = {}
    if 'T_C' in d.columns:
        ren['T_C'] = 'temperature_C'
    if 'RH_pct' in d.columns:
        ren['RH_pct'] = 'rh_pct'
    if 'RH' in d.columns and 'rh_pct' not in d.columns:
        ren['RH'] = 'rh_pct'
    if 'q' in d.columns and 'q_mmol_g' not in d.columns:
        ren['q'] = 'q_mmol_g'
    if 'q_mmol_g_mean' in d.columns and 'q_mmol_g' not in d.columns:
        ren['q_mmol_g_mean'] = 'q_mmol_g'
    d = d.rename(columns=ren)
    if 'temperature_C' not in d.columns and 'temperature_K' in d.columns:
        d['temperature_C'] = d.temperature_K - 273.15
    return d

def clean(df):
    d = normalize_columns(df)
    need = ['temperature_C','rh_pct','q_mmol_g']
    if not set(need).issubset(d.columns):
        return pd.DataFrame(columns=need)
    return (
        d[need]
        .replace([np.inf,-np.inf], np.nan)
        .dropna()
        .groupby(['temperature_C','rh_pct'], as_index=False)['q_mmol_g']
        .mean()
    )

scores = pd.read_csv(SCORES)
required_score_cols = {
    'material','branch','temporal_mode','representation',
    'climate_class','city','n_transitions','H'
}
missing = required_score_cols - set(scores.columns)
if missing:
    raise ValueError(f'Missing Q score columns: {sorted(missing)}')

climate = pd.read_csv(
    CLIMATE,
    usecols=['climate_class','city','timestamp_utc','temperature_C','RH_pct']
)
climate['timestamp_utc'] = pd.to_datetime(
    climate.timestamp_utc, utc=True, errors='coerce'
)
climate = (
    climate.dropna()
    .sort_values(['climate_class','city','timestamp_utc'])
    .reset_index(drop=True)
)

materials = {}

if PEI.exists():
    d = pd.read_csv(PEI)
    if 'material' in d.columns:
        for m,g in d.groupby('material', sort=False):
            z = clean(g)
            if len(z):
                materials[str(m)] = z

raw = pd.read_excel(MASTER, sheet_name='Raw_Data')
mcol = 'Material' if 'Material' in raw.columns else 'material'

for m,g in raw.groupby(mcol, sort=False):
    z = clean(g)
    if len(z) and (str(m) not in materials or len(z) > len(materials[str(m)])):
        materials[str(m)] = z

for pth in [
    PROJECT/'data/processed/validation_IRA_RHT.csv',
    PROJECT/'data/raw/validation_IRA_RHT.csv',
    PROJECT/'data/raw/IRA_A500_RHT.csv',
]:
    if not pth.exists():
        continue
    d = pd.read_csv(pth)
    if 'material' in d.columns:
        for m,g in d.groupby('material', sort=False):
            z = clean(g)
            if len(z) and (str(m) not in materials or len(z) > len(materials[str(m)])):
                materials[str(m)] = z

q_materials = set(scores.material.astype(str).unique())
missing_materials = sorted(q_materials - set(materials))
if missing_materials:
    raise KeyError(
        'Materials present in Q scores but missing from reconstructed equilibrium data: '
        + ', '.join(missing_materials)
    )

print('Q materials:', len(q_materials))
print('Climate rows:', len(climate))


## 2. Reconstruct only full/single support masks

In [ ]:

class FullSurface:
    def __init__(self,d):
        pts = d[['rh_pct','temperature_C']].to_numpy(float)
        vals = d.q_mmol_g.to_numpy(float)
        self.tri = Delaunay(pts)
        self.fun = LinearNDInterpolator(self.tri, vals, fill_value=np.nan)

    def eval(self,rh,T):
        return np.asarray(
            self.fun(np.asarray(rh,float), np.asarray(T,float)),
            float
        )

class SingleIso:
    def __init__(self,d,Tref):
        z = d[d.temperature_C == Tref].sort_values('rh_pct')
        self.Tref = float(Tref)
        self.rmin = float(z.rh_pct.min())
        self.rmax = float(z.rh_pct.max())
        self.fun = interp1d(
            z.rh_pct.to_numpy(float),
            z.q_mmol_g.to_numpy(float),
            kind='linear',
            bounds_error=False,
            fill_value=np.nan
        )

    def eval(self,rh,T):
        return np.asarray(self.fun(np.asarray(rh,float)), float)

class PairSlice:
    def __init__(self,d,T1,T2):
        assert T1 != T2
        self.T1 = float(min(T1,T2))
        self.T2 = float(max(T1,T2))
        z1 = d[d.temperature_C == self.T1].sort_values('rh_pct')
        z2 = d[d.temperature_C == self.T2].sort_values('rh_pct')

        self.rmin = max(float(z1.rh_pct.min()), float(z2.rh_pct.min()))
        self.rmax = min(float(z1.rh_pct.max()), float(z2.rh_pct.max()))

        self.f1 = interp1d(
            z1.rh_pct, z1.q_mmol_g,
            kind='linear', bounds_error=False, fill_value=np.nan
        )
        self.f2 = interp1d(
            z2.rh_pct, z2.q_mmol_g,
            kind='linear', bounds_error=False, fill_value=np.nan
        )

    def eval(self,rh,T):
        rh = np.asarray(rh,float)
        T = np.asarray(T,float)
        q1 = np.asarray(self.f1(rh),float)
        q2 = np.asarray(self.f2(rh),float)

        out = np.full(np.broadcast(rh,T).shape,np.nan,float)
        ok = (T >= self.T1) & (T <= self.T2)
        if np.any(ok):
            lam = (T[ok]-self.T1)/(self.T2-self.T1)
            out[ok] = (1-lam)*q1[ok] + lam*q2[ok]
        return out

class ThreeSlice:
    def __init__(self,d,Tlow,Tref,Thigh):
        self.Tlow = float(Tlow)
        self.Tref = float(Tref)
        self.Thigh = float(Thigh)

        zl = d[d.temperature_C == self.Tlow].sort_values('rh_pct')
        zr = d[d.temperature_C == self.Tref].sort_values('rh_pct')
        zh = d[d.temperature_C == self.Thigh].sort_values('rh_pct')

        self.rmin = max(
            float(zl.rh_pct.min()),
            float(zr.rh_pct.min()),
            float(zh.rh_pct.min())
        )
        self.rmax = min(
            float(zl.rh_pct.max()),
            float(zr.rh_pct.max()),
            float(zh.rh_pct.max())
        )

        self.fl = interp1d(zl.rh_pct,zl.q_mmol_g,bounds_error=False,fill_value=np.nan)
        self.fr = interp1d(zr.rh_pct,zr.q_mmol_g,bounds_error=False,fill_value=np.nan)
        self.fh = interp1d(zh.rh_pct,zh.q_mmol_g,bounds_error=False,fill_value=np.nan)

    def eval(self,rh,T):
        rh = np.asarray(rh,float)
        T = np.asarray(T,float)

        ql = np.asarray(self.fl(rh),float)
        qr = np.asarray(self.fr(rh),float)
        qh = np.asarray(self.fh(rh),float)

        out = np.full(np.broadcast(rh,T).shape,np.nan,float)

        lo = (T >= self.Tlow) & (T <= self.Tref)
        hi = (T >= self.Tref) & (T <= self.Thigh)

        if np.any(lo):
            lam = (T[lo]-self.Tlow)/(self.Tref-self.Tlow)
            out[lo] = (1-lam)*ql[lo] + lam*qr[lo]

        if np.any(hi):
            lam = (T[hi]-self.Tref)/(self.Thigh-self.Tref)
            out[hi] = (1-lam)*qr[hi] + lam*qh[hi]

        return out

# Derive exactly the same Tlow/Tref/Thigh rule used in Q and rebuild the
# four representations that determined Q's branchwise common support.
elig_rows = []
model = {}

for m in sorted(q_materials):
    d = materials[m]
    temps = np.sort(d.temperature_C.unique().astype(float))
    if len(temps) < 3:
        raise ValueError(f'{m}: fewer than 3 temperatures but present in Q scores.')

    Tlow = float(temps[0])
    Thigh = float(temps[-1])
    Tref = float(temps[np.argmin(np.abs(temps - 0.5*(Tlow+Thigh)))])

    elig_rows.append({
        'material': m,
        'Tlow': Tlow,
        'Tref': Tref,
        'Thigh': Thigh,
    })

    model[m] = {
        'full': FullSurface(d),
        'single_ref': SingleIso(d,Tref),
        'nested_low': PairSlice(d,Tlow,Tref),
        'nested_high': PairSlice(d,Tref,Thigh),
        'three_slice': ThreeSlice(d,Tlow,Tref,Thigh),
    }

elig = pd.DataFrame(elig_rows)
display(elig)


## 3. Climate modes and exact Q branch support

In [ ]:

def subsample_climate(df,step):
    out = []
    for (cc,city),g in df.groupby(['climate_class','city'], sort=False):
        z = (
            g.sort_values('timestamp_utc')
            .iloc[::step]
            .copy()
            .reset_index(drop=True)
        )
        out.append(z)
    return pd.concat(out, ignore_index=True)

def average_climate(df,hours):
    out = []
    for (cc,city),g in df.groupby(['climate_class','city'], sort=False):
        z = (
            g.set_index('timestamp_utc')[['temperature_C','RH_pct']]
            .resample(f'{hours}h')
            .mean()
            .dropna()
            .reset_index()
        )
        z['climate_class'] = cc
        z['city'] = city
        out.append(z)
    return pd.concat(out, ignore_index=True)

needed_modes = sorted(scores.temporal_mode.unique())

CLIMATE_MODES = {}
for mode in needed_modes:
    if mode.startswith('subsample_'):
        h = int(mode.split('_')[1].replace('h',''))
        CLIMATE_MODES[mode] = subsample_climate(climate,h)
    elif mode.startswith('average_'):
        h = int(mode.split('_')[1].replace('h',''))
        CLIMATE_MODES[mode] = average_climate(climate,h)
    else:
        raise ValueError(f'Unknown temporal mode: {mode}')

def base_transitions(cdf,Tlo,Thi,Rlo,Rhi):
    out = []
    for (cc,city),g in cdf.groupby(['climate_class','city'], sort=False):
        g = g.sort_values('timestamp_utc').reset_index(drop=True)
        rh = g.RH_pct.to_numpy(float)
        T = g.temperature_C.to_numpy(float)
        ts = g.timestamp_utc.to_numpy()

        ok = (
            (rh[:-1] >= Rlo) & (rh[:-1] <= Rhi) &
            (rh[1:] >= Rlo) & (rh[1:] <= Rhi) &
            (T[:-1] >= Tlo) & (T[:-1] <= Thi) &
            (T[1:] >= Tlo) & (T[1:] <= Thi)
        )
        ix = np.where(ok)[0]
        if len(ix):
            out.append(pd.DataFrame({
                'transition_id': [
                    f'{cc}|{city}|{pd.Timestamp(ts[k]).isoformat()}'
                    for k in ix
                ],
                'climate_class': cc,
                'city': city,
                'RH0': rh[ix],
                'RH1': rh[ix+1],
                'T0': T[ix],
                'T1': T[ix+1],
            }))
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


def exact_q_common_mask(base, md, branch):
    # Q used the intersection of ALL branch representations, not merely full+single.
    reps = (
        ['full','single_ref','nested_low','three_slice']
        if branch == 'lower'
        else ['full','single_ref','nested_high','three_slice']
    )

    masks = []
    for rep in reps:
        obj = md[rep]
        q0 = obj.eval(
            base.RH0.to_numpy(float),
            base.T0.to_numpy(float)
        )
        q1 = obj.eval(
            base.RH1.to_numpy(float),
            base.T1.to_numpy(float)
        )
        masks.append(np.isfinite(q0) & np.isfinite(q1))

    return np.logical_and.reduce(masks)

def exact_q_R_support(md, branch):
    reps = (
        ['full','single_ref','nested_low','three_slice']
        if branch == 'lower'
        else ['full','single_ref','nested_high','three_slice']
    )

    rmins, rmaxs = [], []
    for rep in reps:
        obj = md[rep]
        if hasattr(obj,'rmin'):
            rmins.append(float(obj.rmin))
            rmaxs.append(float(obj.rmax))

    return max([0.0]+rmins), min([100.0]+rmaxs)



## 4. Build the RH-trajectory null on the exact same transition set as Q

For each material × branch × temporal mode:

1. use the target branch limits;
2. use Q’s branchwise common RH support across single, nested-two-slice, and three-slice representations;
3. reconstruct the exact Q finite-support intersection of full, single, nested-two-slice, and three-slice representations;
4. compute
   \[
   H_{\rm RH}=\frac12\sum |\Delta RH|
   \]
   on those transitions;
5. join to Q's already-computed `full` and `single_ref` climate scores;
6. **require exact equality of \(N\)** between the reconstructed null table and Q.

If \(N\) does not match, the notebook stops for that problem rather than silently comparing different supports.


In [ ]:

MIN_TRANSITIONS = 12

null_rows = []
audit_rows = []

elig_map = elig.set_index('material').to_dict('index')

for (m,branch,mode), gq in tqdm(
    scores.groupby(['material','branch','temporal_mode'], sort=False),
    desc='Reconstruct exact RH-null support'
):
    er = elig_map[m]
    Tlo, Tref, Thigh = er['Tlow'], er['Tref'], er['Thigh']

    if branch == 'lower':
        Blo, Bhi = Tlo, Tref
    elif branch == 'upper':
        Blo, Bhi = Tref, Thigh
    else:
        continue

    md = model[m]

    # EXACT Q RH support: intersection across every representation used in
    # that branch's Q hierarchy.
    Rlo, Rhi = exact_q_R_support(md, branch)

    base = base_transitions(
        CLIMATE_MODES[mode],
        Blo, Bhi, Rlo, Rhi
    )
    if base.empty:
        continue

    # EXACT Q finite-support mask: full ∩ single ∩ nested branch ∩ three-slice.
    mask = exact_q_common_mask(base, md, branch)
    z = base.loc[mask].copy()

    z['abs_dRH_half'] = 0.5*np.abs(z.RH1-z.RH0)

    nul = (
        z.groupby(['climate_class','city'], as_index=False)
        .agg(
            N_null=('transition_id','size'),
            H_RH_null=('abs_dRH_half','sum')
        )
    )
    nul = nul[nul.N_null >= MIN_TRANSITIONS].copy()

    qf = (
        gq[gq.representation=='full']
        [['climate_class','city','n_transitions','H']]
        .rename(columns={'n_transitions':'N_Q_full','H':'H_full'})
    )
    qs = (
        gq[gq.representation=='single_ref']
        [['climate_class','city','n_transitions','H']]
        .rename(columns={'n_transitions':'N_Q_single','H':'H_single'})
    )

    merged = (
        qf.merge(qs,on=['climate_class','city'],how='inner')
        .merge(nul,on=['climate_class','city'],how='inner')
    )

    if merged.empty:
        continue

    same_N = (
        (merged.N_Q_full == merged.N_Q_single) &
        (merged.N_Q_full == merged.N_null)
    )

    audit_rows.append({
        'material': m,
        'branch': branch,
        'temporal_mode': mode,
        'Rlo_Q': Rlo,
        'Rhi_Q': Rhi,
        'n_climates_joined': len(merged),
        'n_climates_exact_N_match': int(same_N.sum()),
        'fraction_exact_N_match': float(same_N.mean()),
        'N_Q_full_min': int(merged.N_Q_full.min()),
        'N_Q_full_max': int(merged.N_Q_full.max()),
    })

    if not same_N.all():
        bad = merged.loc[~same_N,[
            'climate_class','city','N_Q_full','N_Q_single','N_null'
        ]].copy()

        bad.to_csv(
            OUT/f'S2_SUPPORT_MISMATCH_{m}_{branch}_{mode}.csv'
                .replace('/','_').replace(' ','_'),
            index=False
        )

        raise RuntimeError(
            f'Exact Q-support reconstruction still failed for {m} | {branch} | {mode}. '
            f'{len(bad)} climates have mismatched N. '
            'A mismatch CSV was saved for diagnosis.'
        )

    merged['material'] = m
    merged['branch'] = branch
    merged['temporal_mode'] = mode
    merged['N'] = merged.N_Q_full.astype(int)

    null_rows.append(merged[[
        'material','branch','temporal_mode',
        'climate_class','city','N',
        'H_full','H_single','H_RH_null'
    ]])

gate_data = pd.concat(null_rows, ignore_index=True)
audit = pd.DataFrame(audit_rows)

gate_data.to_csv(OUT/'S2_gate_climate_level.csv', index=False)
audit.to_csv(OUT/'S2_support_reconstruction_audit.csv', index=False)

print('Climate-level rows:', len(gate_data))
print('Problems:', gate_data.groupby(['material','branch','temporal_mode']).ngroups)
print('Minimum support-match fraction:', audit.fraction_exact_N_match.min())
display(audit.head(30))


## 5. Point estimates

In [ ]:

EPS = 1e-12

def safe_spearman(x,y):
    x = np.asarray(x,float)
    y = np.asarray(y,float)
    if len(x) < 4 or np.unique(x).size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(spearmanr(x,y).statistic)

def partial_spearman(x,y,n):
    rxy = safe_spearman(x,y)
    rxn = safe_spearman(x,n)
    ryn = safe_spearman(y,n)
    if not np.all(np.isfinite([rxy,rxn,ryn])):
        return np.nan
    den = np.sqrt(max(EPS,1-rxn**2)*max(EPS,1-ryn**2))
    return float((rxy-rxn*ryn)/den)

point = []

for (m,branch,mode),g in gate_data.groupby(
    ['material','branch','temporal_mode'], sort=False
):
    F = g.H_full.to_numpy(float)
    S = g.H_single.to_numpy(float)
    R = g.H_RH_null.to_numpy(float)
    N = g.N.to_numpy(float)

    rs = partial_spearman(S,F,N)
    rr = partial_spearman(R,F,N)

    point.append({
        'material':m,
        'branch':branch,
        'temporal_mode':mode,
        'n_climates':len(g),

        'tau_single_full_raw':kendalltau(S,F).statistic,
        'tau_RHnull_full_raw':kendalltau(R,F).statistic,

        'rho_single_full_raw':safe_spearman(S,F),
        'rho_RHnull_full_raw':safe_spearman(R,F),

        'rho_single_full_partial_N':rs,
        'rho_RHnull_full_partial_N':rr,
        'delta_rho_partial':rs-rr,

        'rho_full_N':safe_spearman(F,N),
        'rho_single_N':safe_spearman(S,N),
        'rho_RHnull_N':safe_spearman(R,N),
    })

point = pd.DataFrame(point)
point.to_csv(OUT/'S2_point_estimates.csv', index=False)
display(point.sort_values(['temporal_mode','branch','material']))



## 6. Optimized paired climate bootstrap

The expensive equilibrium interpolation is already finished before this section.

Bootstrap operates only on small climate-level arrays.

Optimization:
- resampling indices are generated in batches;
- rank transforms use `scipy.stats.rankdata(..., axis=1)`;
- Pearson correlations on ranks are computed row-wise with NumPy;
- no pandas groupby, no interpolation, and no repeated `scipy.stats.spearmanr` calls occur inside individual bootstrap replicates.

This retains exact Spearman logic, including average ranks for ties.


In [ ]:

N_BOOT = 5000
BATCH = 500
SEED = 20261013
rng = np.random.default_rng(SEED)

def row_corr(a,b):
    a = np.asarray(a,float)
    b = np.asarray(b,float)
    ac = a - a.mean(axis=1,keepdims=True)
    bc = b - b.mean(axis=1,keepdims=True)
    num = np.sum(ac*bc,axis=1)
    den = np.sqrt(np.sum(ac*ac,axis=1)*np.sum(bc*bc,axis=1))
    out = np.full(len(a),np.nan,float)
    ok = den > 1e-14
    out[ok] = num[ok]/den[ok]
    return out

def partial_from_rank_matrices(X,Y,N):
    rxy = row_corr(X,Y)
    rxn = row_corr(X,N)
    ryn = row_corr(Y,N)
    den = np.sqrt(
        np.maximum(0.0,1-rxn**2) *
        np.maximum(0.0,1-ryn**2)
    )
    out = np.full(len(rxy),np.nan,float)
    ok = den > 1e-14
    out[ok] = (rxy[ok]-rxn[ok]*ryn[ok])/den[ok]
    return out

def bootstrap_partial_delta(F,S,R,N,n_boot=N_BOOT,batch=BATCH):
    F = np.asarray(F,float)
    S = np.asarray(S,float)
    R = np.asarray(R,float)
    N = np.asarray(N,float)
    n = len(F)

    all_rs, all_rr, all_d = [], [], []

    done = 0
    while done < n_boot:
        b = min(batch,n_boot-done)
        ix = rng.integers(0,n,size=(b,n))

        Fb = F[ix]
        Sb = S[ix]
        Rb = R[ix]
        Nb = N[ix]

        # Exact Spearman ranks row-wise.
        Fr = rankdata(Fb,axis=1,method='average')
        Sr = rankdata(Sb,axis=1,method='average')
        Rr = rankdata(Rb,axis=1,method='average')
        Nr = rankdata(Nb,axis=1,method='average')

        rs = partial_from_rank_matrices(Sr,Fr,Nr)
        rr = partial_from_rank_matrices(Rr,Fr,Nr)
        d = rs-rr

        ok = np.isfinite(rs)&np.isfinite(rr)&np.isfinite(d)
        all_rs.append(rs[ok])
        all_rr.append(rr[ok])
        all_d.append(d[ok])

        done += b

    return (
        np.concatenate(all_rs),
        np.concatenate(all_rr),
        np.concatenate(all_d)
    )

ci_rows = []

for (m,branch,mode),g in tqdm(
    gate_data.groupby(['material','branch','temporal_mode'], sort=False),
    desc='Vectorized climate bootstrap'
):
    rs, rr, dd = bootstrap_partial_delta(
        g.H_full.to_numpy(float),
        g.H_single.to_numpy(float),
        g.H_RH_null.to_numpy(float),
        g.N.to_numpy(float)
    )

    if len(dd) < 500:
        continue

    qs = np.quantile(rs,[.025,.5,.975])
    qr = np.quantile(rr,[.025,.5,.975])
    qd = np.quantile(dd,[.025,.5,.975])

    ci_rows.append({
        'material':m,
        'branch':branch,
        'temporal_mode':mode,
        'n_climates':len(g),
        'n_boot_valid':len(dd),

        'rho_single_partial_lo':qs[0],
        'rho_single_partial_median':qs[1],
        'rho_single_partial_hi':qs[2],

        'rho_RHnull_partial_lo':qr[0],
        'rho_RHnull_partial_median':qr[1],
        'rho_RHnull_partial_hi':qr[2],

        'delta_rho_lo':qd[0],
        'delta_rho_median':qd[1],
        'delta_rho_hi':qd[2],

        'bootstrap_prob_delta_gt_0':float(np.mean(dd>0)),
        'strict_CI_pass':bool(qd[0]>0),
    })

ci = pd.DataFrame(ci_rows)
ci.to_csv(OUT/'S2_climate_bootstrap_CI.csv',index=False)

result = point.merge(
    ci,
    on=['material','branch','temporal_mode','n_climates'],
    how='left'
)
result.to_csv(OUT/'S2_gate_by_material.csv',index=False)

display(result.sort_values(['temporal_mode','branch','material']))


## 7. Gate summary

In [ ]:

SUBDAILY = ['subsample_1h','subsample_3h','subsample_6h']

summary = (
    result.groupby(['branch','temporal_mode'],as_index=False)
    .agg(
        n_materials=('material','nunique'),
        median_rho_single_partial_N=('rho_single_full_partial_N','median'),
        median_rho_RHnull_partial_N=('rho_RHnull_full_partial_N','median'),
        median_delta_rho=('delta_rho_partial','median'),
        fraction_delta_positive=(
            'delta_rho_partial',
            lambda x:float(np.mean(np.asarray(x,float)>0))
        ),
        fraction_strict_CI_pass=(
            'strict_CI_pass',
            lambda x:float(np.mean(pd.Series(x).fillna(False).astype(bool)))
        ),
    )
)

summary['subdaily'] = summary.temporal_mode.isin(SUBDAILY)
summary.to_csv(OUT/'S2_gate_summary.csv',index=False)

display(summary)
print('\nSub-daily only:')
display(
    summary[summary.subdaily]
    .sort_values(['branch','temporal_mode'])
)



## Interpretation

### Gate remains open
A narrow claim is supportable if the 1/3/6 h results show that:

\[
\rho_{SF\cdot N} > \rho_{RH,F\cdot N}
\]

for most materials, with positive \(\Delta\rho\) and climate-bootstrap CIs above zero for a meaningful fraction of the panel.

Candidate wording:

> Single-isotherm climate rankings retain material-specific agreement with the full equilibrium representation beyond supported-transition occupancy and a shared-RH-trajectory null at sub-daily resolution.

### Gate closes
If the RH-null matches the single-isotherm partial correlation, or if \(\Delta\rho\) is centered near zero with broad CIs, then the remaining single–full agreement cannot be attributed confidently to target-material equilibrium information.

### Scope
Passing this gate does **not** establish:
- full process fidelity,
- productivity,
- deployment decisions,
- cross-material sorbent ranking,
- universal single-isotherm sufficiency.

It establishes only that the single target-material isotherm contains nontrivial climate-ranking information beyond transition count and generic RH trajectory structure.
